In [1]:

from sklearn.preprocessing import LabelEncoder
import seaborn as sns

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

solar_d = pd.read_csv(
    '../data/fact_solar_daily.csv',
    parse_dates=['date']
)

solar_h = pd.read_csv(
    '../data/fact_solar_hourly.csv',
    parse_dates=['date', 'hour_ts']
)

weather_d = pd.read_csv(
    '../data/fact_weather_daily.csv',
    parse_dates=['date']
)

weather_h = pd.read_csv(
    '../data/fact_weather_hourly.csv',
    parse_dates=['date', 'hour_ts']
)

dim_date = pd.read_csv(
    '../data/dim_date.csv',
    parse_dates=['date']
)

wc_codes = pd.read_csv(
    '../data/dim_weather_codes.csv'
)

# Merge daily solar + weather
daily = pd.merge(
    solar_d,
    weather_d,
    on='date',
    how='inner'
)

# Add date dimension columns
daily = pd.merge(
    daily,
    dim_date[['date', 'month_name', 'season', 'is_weekend', 'day_name']],
    on='date',
    how='left'
)

print(daily.shape)


(89, 20)


In [14]:
df = daily.copy()

# Encode season (Dry=0, Wet=1)
df['season_enc']      = (df['season'] == 'Wet').astype(int)
df['is_weekend_enc'] = df['is_weekend'].fillna(False).astype(int)

# Sunshine efficiency ratio
df['sunshine_ratio'] = df['sunshine_duration'] / (df['daylight_duration'] + 1e-5)

# Radiation per cloud (interaction feature)
df['rad_clear'] = df['shortwave_radiation_sum'] * (1 - df['cloud_cover_mean']/100)

FEATURES = [
    'shortwave_radiation_sum', 'sunshine_duration',
    'cloud_cover_mean', 'temperature_2m_mean',
    'wind_speed_10m_mean', 'rain_sum',
    'season_enc', 'is_weekend_enc',
    'sunshine_ratio', 'rad_clear'
]
TARGET = 'generation_kwh'

X = df[FEATURES]
y = df[TARGET]


/var/folders/lh/gvpbf1b15yd43l_x_kk3zxgc0000gn/T/ipykernel_31993/425362064.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_weekend_enc'] = df['is_weekend'].fillna(False).astype(int)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f'Train: {len(X_train)}')
print(f'Test : {len(X_test)}')

Train: 71
Test : 18


In [18]:

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# --- Model 1: Ridge Regression (baseline) ---
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])
ridge_pipe.fit(X_train, y_train)

# --- Model 2: Random Forest (same as reference repo) ---
rf_params = {
    'n_estimators':     [100, 200],
    'max_depth':        [None, 10, 20],
    'min_samples_split':[2, 5],
}
rf_cv = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params, cv=5, scoring='r2', n_jobs=-1
)
rf_cv.fit(X_train, y_train)
best_rf = rf_cv.best_estimator_
print('Best RF params:', rf_cv.best_params_)

# --- Model 3: Gradient Boosting ---
gb_params = {'n_estimators':[100,200],'learning_rate':[0.05,0.1],'max_depth':[3,5]}
gb_cv = GridSearchCV(GradientBoostingRegressor(random_state=42),
                     gb_params, cv=5, scoring='r2', n_jobs=-1)
gb_cv.fit(X_train, y_train)

# --- Evaluation ---
for name, model in [('Ridge', ridge_pipe), ('RF', best_rf), ('GB', gb_cv.best_estimator_)]:
    pred = model.predict(X_test)
    print(f'{name}  R²={r2_score(y_test,pred):.4f}  MAE={mean_absolute_error(y_test,pred):.2f}  RMSE={np.sqrt(mean_squared_error(y_test,pred)):.2f}')

Best RF params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
Ridge  R²=0.4885  MAE=3.91  RMSE=6.71
RF  R²=0.4446  MAE=4.15  RMSE=7.00
GB  R²=0.5141  MAE=4.15  RMSE=6.54


In [11]:
# Encode season
df['season_enc'] = (df['season'] == 'Wet').astype(int)

# Encode weekend
df['is_weekend_enc'] = df['is_weekend'].astype(bool).astype(int)

# Sunshine efficiency
df['sunshine_ratio'] = (
    df['sunshine_duration'] /
    (df['daylight_duration'] + 1e-5)
)

# Radiation adjusted by cloud cover
df['rad_clear'] = (
    df['shortwave_radiation_sum'] *
    (1 - df['cloud_cover_mean'] / 100)
)

In [ ]:
import joblib

# Save best model
joblib.dump(
    gb_cv.best_estimator_,
    '../models/solar_generation_model.pkl'
)

# Save feature names
joblib.dump(
    FEATURES,
    '../models/feature_names.pkl'
)

print("Model and feature list saved.")

Model and feature list saved.


Exception ignored in: <function ResourceTracker.__del__ at 0x10a1b6660>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10beee660>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.